# cancer-recog-apoptosis — RUNG 4 / Step-5: logic-gate recognition designer

Searches **AND / AND-NOT** antigen combinations and scores each by single-cell selectivity — *is any normal cell, especially heart/brain/kidney, double-positive?* — with **fail-closed** vital safety (a vital organ that isn't captured → UNCERTAIN, never silently SAFE).

**CPU runtime (no GPU).** The real Census fetch is ~50 min (brain alone is 10.5M cells). To survive a disconnect/laptop-sleep, Cell 2 mounts **Google Drive** and caches the fetched panel there — so if it dies, re-running **resumes in seconds** instead of re-fetching. Run once, keep the tab open if you can; if it disconnects, just Run-all again.

In [ ]:
#@title Cell 1 — clone / pull repo
import os
from pathlib import Path
REPO = Path('/content/cancer-recog-apoptosis')
if REPO.exists():
    !cd {REPO} && rm -f runs/logs/*.log && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recog-apoptosis.git {REPO}
os.chdir(REPO)
!git log --oneline -1
print('[CELL 1] ✓')

In [ ]:
#@title Cell 2 — mount Google Drive for a RESUMABLE cache (recommended) + start run log
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    cache_dir = '/content/drive/MyDrive/cancer-recon'
    os.makedirs(cache_dir, exist_ok=True)
    os.environ['LOGICGATE_CACHE'] = f'{cache_dir}/rung4_panel_cache.npz'
    print('[CELL 2] Drive mounted — fetch will be cached to', os.environ['LOGICGATE_CACHE'])
    print('         If the laptop sleeps mid-run, just Run-all again: it resumes from this cache (no re-fetch).')
except Exception as e:
    os.environ['LOGICGATE_CACHE'] = '/content/cancer-recog-apoptosis/data/logicgate/rung4_panel_cache.npz'
    print('[CELL 2] Drive NOT mounted (', type(e).__name__, ') — caching to /content (LOST on disconnect).')
from scripts.runlog import new_log, run_logged, finalize
RUNLOG = new_log('rung4_logicgate', repo_dir='.')
print('[CELL 2] ✓')

In [ ]:
#@title Cell 3 — VALIDATE the engine on biological ground truth (CPU, no data needed)
import sys
from scripts.runlog import new_log, run_logged
RUNLOG = globals().get('RUNLOG') or new_log('rung4_logicgate', repo_dir='.')
rc = run_logged([sys.executable, '-u', 'scripts/20_logicgate_calibration.py'], RUNLOG)
print(f'[CELL 3] exit={rc}', '✓ RUN-TRUST passed' if rc == 0 else '✗ engine failed validation')
from IPython.display import Image, display
import os
if os.path.exists('runs/rung4_logicgate/rung4_calibration.png'):
    display(Image('runs/rung4_logicgate/rung4_calibration.png'))

In [ ]:
#@title Cell 4 — install CELLxGENE Census (only needed for the real discovery)
import sys, importlib.util
from scripts.runlog import run_logged
for pkg, pip_name in [('cellxgene_census', 'cellxgene-census'), ('scanpy', 'scanpy')]:
    if importlib.util.find_spec(pkg) is None:
        run_logged([sys.executable, '-m', 'pip', 'install', '-q', pip_name], RUNLOG, label=f'pip install {pip_name}')
print('[CELL 4] ✓ (if Colab asks to RESTART runtime, do it, then Run all again — the cache makes that cheap)')

In [ ]:
#@title Cell 5 — REAL discovery on CELLxGENE atlases (~50 min first run; SECONDS if resuming from cache)
import sys
rc = run_logged([sys.executable, '-u', 'scripts/17_logicgate_data.py'], RUNLOG)
print(f'[CELL 5] exit={rc}')
import json, os
if os.path.exists('runs/rung4_logicgate/rung4_results.json'):
    r = json.load(open('runs/rung4_logicgate/rung4_results.json'))
    print('vital coverage:', r.get('vital_coverage'))
    print('UNAUDITED vital types:', r.get('unaudited_vital_types'))
    print('gates scored:', r.get('n_gates'), '| predicted SELECTIVE:', r.get('n_selective'))
    print('NO clean gate?', r.get('no_clean_gate'))
from IPython.display import Image, display
if os.path.exists('runs/rung4_logicgate/rung4_discovery.png'):
    display(Image('runs/rung4_logicgate/rung4_discovery.png'))

In [ ]:
#@title Cell 6 — finalize + download run log AND all figures / result files
import os, glob
from scripts.runlog import new_log, finalize
RUNLOG = globals().get('RUNLOG') or new_log('rung4_logicgate', repo_dir='.')
finalize(RUNLOG)   # downloads the commit-stamped log to your Downloads folder
EXTRA = (glob.glob('runs/rung4_logicgate/*.png') + glob.glob('runs/rung4_logicgate/*.json')
         + glob.glob('runs/rung4_logicgate/*.csv') + glob.glob('data/logicgate/*.csv'))
try:
    from google.colab import files
    for p in sorted(set(EXTRA)):
        if os.path.exists(p):
            print('[download]', p); files.download(p)
except ImportError:
    print('(not in Colab — download skipped)')
except Exception as e:
    print('(download skipped:', type(e).__name__, e, ')')

## What this rung established

- The selectivity engine is validated (re-derives Step-3 HER2-unsafe-on-heart; exposes the bulk-trap; FAILS CLOSED on missing vital tissue).
- Real discovery is transcript-only (CITE-seq confirms co-positivity); leaks are Jeffreys upper bounds; vital cells kept up to 50k each; multiple-testing (held-out-donor) deferred → selective gates are a DISCOVERY shortlist.
- 'No clean gate exists' is a first-class result.

Next: snRNA-seq for under-sampled vital types (adrenal/muscle), CITE-seq protein cross-check, and the clonal antigen-loss durability sim.